<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-08-agents-and-adk/lesson-8.1-root-agent/notebooks/GCP_Capstone_8.1_RootAgent.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8.1 Root Agent with ADK
**Netsetos GenAI Engineering — GCP Capstone**


In [ ]:
!pip install -q google-adk google-genai
import os
from google.colab import auth
auth.authenticate_user()
os.environ['GOOGLE_CLOUD_PROJECT'] = 'YOUR-PROJECT'
os.environ['GOOGLE_CLOUD_LOCATION'] = 'global'  # global endpoint for Gemini 3.x generation
os.environ['GOOGLE_GENAI_USE_VERTEXAI'] = 'TRUE'
print('ADK ready (Vertex + ADC)')


## Cell 1: Define Tools


In [ ]:
from google.adk.tools import ToolContext

def search_documents(query: str, tool_context: ToolContext) -> dict:
    """Search the document knowledge base.
    Args:
        query: Search query to find relevant documents.
    """
    history = tool_context.state.get('search_history', [])
    history.append(query)
    tool_context.state['search_history'] = history
    return {'results': [{'id': 'D-01', 'title': 'Q1 Report', 'relevance': 0.95}]}

def summarize_document(document_id: str, summary_type: str, tool_context: ToolContext) -> dict:
    """Summarize a specific document.
    Args:
        document_id: Document ID.
        summary_type: brief, detailed, or executive.
    """
    tool_context.state['last_summarized'] = document_id
    return {'summary': f'Summary of {document_id}...'}

def calculate_cost(page_count: int, tier: str) -> dict:
    """Calculate document processing cost.
    Args:
        page_count: Number of pages.
        tier: standard, premium, enterprise.
    """
    rates = {'standard': 0.01, 'premium': 0.03, 'enterprise': 0.05}
    return {'cost_usd': round(page_count * rates.get(tier, 0.01), 2)}

print('3 tools defined')


## Cell 2: Create Root Agent


In [ ]:
from google.adk.agents import LlmAgent
from google.genai import types

root_agent = LlmAgent(
    name='documind', model='gemini-3.6-flash',
    instruction='You are DocuMind AI. Use search_documents to find docs. '
                'Use summarize_document for summaries. Use calculate_cost for pricing. '
                'Always search before summarizing.',
    generate_content_config=types.GenerateContentConfig(temperature=0.2),
    tools=[search_documents, summarize_document, calculate_cost],
)
print(f'Agent: {root_agent.name}')


## Cell 3: Run with Runner


In [ ]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai.types import Content, Part

async def test_agent():
    ss = InMemorySessionService()
    runner = Runner(agent=root_agent, app_name='dm', session_service=ss)
    session = await ss.create_session(app_name='dm', user_id='s1')
    for msg in ['Find financial documents', 'Summarize the first result',
                'Cost for 100 pages premium?']:
        print(f'\nUser: {msg}')
        m = Content(role='user', parts=[Part(text=msg)])
        async for ev in runner.run_async(user_id='s1', session_id=session.id, new_message=m):
            if ev.is_final_response() and ev.content and ev.content.parts:
                for p in ev.content.parts:
                    if p.text: print(f'Agent: {p.text[:150]}')
    s = await ss.get_session(app_name='dm', user_id='s1', session_id=session.id)
    print(f'\nState: {dict(s.state)}')

await test_agent()


## Cell 4: Write agent.py for adk web


In [ ]:
import os
os.makedirs('documind_agent', exist_ok=True)
with open('documind_agent/__init__.py','w') as f: f.write('from . import agent\n')
agent_py = '''from google.adk.agents import LlmAgent
from google.adk.tools import ToolContext
from google.genai import types

def search_documents(query: str, tool_context: ToolContext) -> dict:
    """Search documents.\n    Args:\n        query: Search query.\n    """
    h = tool_context.state.get("search_history", [])
    h.append(query)
    tool_context.state["search_history"] = h
    return {"results": [{"id": "D-01", "title": "Q1 Report"}]}

def summarize_document(document_id: str, summary_type: str, tool_context: ToolContext) -> dict:
    """Summarize a document.\n    Args:\n        document_id: Doc ID.\n        summary_type: brief/detailed/executive.\n    """
    tool_context.state["last_summarized"] = document_id
    return {"summary": f"Summary of {document_id}"}

def calculate_cost(page_count: int, tier: str) -> dict:
    """Calculate processing cost.\n    Args:\n        page_count: Pages.\n        tier: standard/premium/enterprise.\n    """
    rates = {"standard": 0.01, "premium": 0.03, "enterprise": 0.05}
    return {"cost_usd": round(page_count * rates.get(tier, 0.01), 2)}

root_agent = LlmAgent(
    name="documind", model="gemini-3.6-flash",
    instruction="You are DocuMind AI. Search before summarizing.",
    generate_content_config=types.GenerateContentConfig(temperature=0.2),
    tools=[search_documents, summarize_document, calculate_cost],
)
'''
with open('documind_agent/agent.py','w') as f: f.write(agent_py)
with open('documind_agent/.env','w') as f: f.write('GOOGLE_API_KEY=YOUR-KEY\n')
print('Agent package created. Run: cd .. && adk web')


## Done!
- LlmAgent with instruction + tools
- ToolContext for session state
- Runner event loop
- adk web project structure
